# 06 — H-FraudGT: FraudGT với đặc trưng hành vi lịch sử

Notebook thực nghiệm kế tiếp sau baseline A/B. Mục tiêu là kiểm tra riêng tác động của **History Features (H)** trước khi thay đổi neighbor sampler.

- Giữ nguyên backbone FraudGT đầy đủ: RMP + Ports + Ego IDs + 2 FraudGT Encoder blocks.
- Giữ nguyên original hetero-edge head, loss, optimizer, sampler và ngân sách huấn luyện của baseline A.
- Chỉ thêm 8 đặc trưng được tính từ các giao dịch có timestamp **nhỏ hơn nghiêm ngặt** giao dịch mục tiêu.
- Chạy một seed 42. Không dùng test để chọn epoch hoặc threshold.

`RUN_MODE='full'` chạy 100 epoch để lấy kết quả báo cáo. Đổi thành `'smoke'` nếu chỉ muốn kiểm tra pipeline 2 epoch.


In [ ]:
RUN_MODE = 'full'  # 'smoke' hoặc 'full'
SEED = 42
GPU = 0
assert RUN_MODE in {'smoke', 'full'}
print('RUN_MODE:', RUN_MODE, '| seed:', SEED, '| GPU:', GPU)


## 1. Ghi nhận môi trường


In [ ]:
import platform, sys, subprocess, torch
print('Python:', sys.version)
print('Platform:', platform.platform())
print('PyTorch:', torch.__version__)
print('CUDA runtime:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name}; VRAM={p.total_memory / 1024**3:.2f} GiB')
subprocess.run(['nvidia-smi'], check=False)


## 2. Cài dependency


In [ ]:
import subprocess, sys, torch
torch_version = torch.__version__.split('+')[0]
cuda_tag = 'cu' + torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'
wheel_url = f'https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html'
print('PyG wheel index:', wheel_url)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pyg_lib', 'torch_scatter', 'torch_sparse', '-f', wheel_url], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch_geometric', 'torchmetrics', 'yacs', 'datatable',
                'pandas', 'matplotlib', 'wandb', 'ogb', 'tensorboardX',
                'pyyaml'], check=True)
print('Dependencies installed.')


## 3. Lấy đúng repository và ghi commit


In [ ]:
from pathlib import Path
import os, subprocess

REPO_URL = 'https://github.com/mhiunguyen/MV-IA-FraudGT.git'
repo = Path('/kaggle/working/MV-IA-FraudGT')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
os.chdir(repo)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Repository:', repo)
print('Commit:', commit)

required = [
    repo / 'fraudGT' / 'datasets' / 'history_features.py',
    repo / 'configs' / 'AML-Small-HI' / 'AML-Small-HI-History-T4.yaml',
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise RuntimeError('Repository chưa chứa mã H-FraudGT. Hãy push/pull commit mới. Thiếu: ' + str(missing))


## 4. Gắn AML-Small-HI


In [ ]:
from shutil import copy2

candidates = list(Path('/kaggle/input').rglob('HI-Small_Trans.csv'))
if not candidates:
    raise FileNotFoundError(
        'Không tìm thấy HI-Small_Trans.csv. Hãy Add Input bộ IBM Transactions for AML.'
    )
source = candidates[0]
destination = repo / 'data' / 'AML' / 'HI-Small_Trans.csv'
destination.parent.mkdir(parents=True, exist_ok=True)
if not destination.exists() or destination.stat().st_size != source.stat().st_size:
    copy2(source, destination)
print('Nguồn:', source)
print('Đích:', destination)
print(f'Dung lượng: {destination.stat().st_size / 1024**2:.1f} MiB')


## 5. Kiểm tra chống temporal leakage trên ví dụ nhỏ

Hai giao dịch có cùng timestamp không được nhìn thấy nhau. Cell dưới phải hoàn thành trước khi xử lý toàn bộ dữ liệu.


In [ ]:
import numpy as np
import pandas as pd
import sys
sys.path.insert(0, str(repo))
from fraudGT.datasets.history_features import (
    HISTORY_FEATURE_NAMES,
    compute_past_only_history_features_raw,
    normalize_history_features_train_only,
)

toy = pd.DataFrame({
    'from_id': [0, 0, 0, 3],
    'to_id': [1, 2, 1, 1],
    'Timestamp': [10, 10, 20, 20],
    'Amount Received': [10.0, 20.0, 30.0, 5.0],
})
toy_history = compute_past_only_history_features_raw(toy)
toy_table = pd.DataFrame(toy_history, columns=HISTORY_FEATURE_NAMES)
display(pd.concat([toy, toy_table], axis=1))

assert np.allclose(toy_history[0], 0.0)
assert np.allclose(toy_history[1], 0.0), 'Cạnh cùng timestamp đã nhìn thấy nhau.'
assert np.isclose(toy_history[2, 4], np.log1p(2))
assert np.isclose(toy_history[2, 6], np.log1p(1))
print('LEAKAGE TEST: PASS — chỉ sử dụng timestamp nhỏ hơn nghiêm ngặt.')


## 6. Tạo config thực tế


In [ ]:
import copy, yaml

base_cfg_path = repo / 'configs' / 'AML-Small-HI' / 'AML-Small-HI-History-T4.yaml'
cfg = yaml.safe_load(base_cfg_path.read_text(encoding='utf-8'))
cfg['out_dir'] = str(repo / 'results')
cfg['dataset']['dir'] = str(repo / 'data')
cfg['seed'] = SEED

if RUN_MODE == 'smoke':
    cfg['train']['iter_per_epoch'] = 16
    cfg['val']['iter_per_epoch'] = 32
    cfg['train']['eval_period'] = 1
    cfg['optim']['max_epoch'] = 2

config_dir = Path('/kaggle/working/generated_configs')
config_dir.mkdir(parents=True, exist_ok=True)
RUN_CFG = config_dir / f'AML-Small-HI-History-{RUN_MODE}.yaml'
RUN_CFG.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
TAG = f'H-History-{RUN_MODE.capitalize()}'
print('Config:', RUN_CFG)
print('Tag:', TAG)
print(RUN_CFG.read_text(encoding='utf-8'))


## 7. Tiền xử lý H-FraudGT và kiểm tra cache

Lần đầu có thể mất thời gian vì phải tính lịch sử trên toàn bộ giao dịch và tạo Ports. Cache H dùng tên riêng `data_history.pt`/`ports_history.pt`, không ghi đè cache baseline A.


In [ ]:
import gc, time, torch
from fraudGT.datasets.aml_dataset import AMLDataset

started = time.time()
dataset_cache = AMLDataset(
    root=str(repo / 'data' / 'AML'), name='Small-HI',
    reverse_mp=True, add_ports=True, add_history=True,
)
print('Processed files:', dataset_cache.processed_paths)
print(f'Preprocessing/cache load: {(time.time() - started) / 60:.1f} minutes')

expected_dim = 4 + len(HISTORY_FEATURE_NAMES) + 1  # raw edge + history + one port/relation
for split in ['train', 'val', 'test']:
    data = dataset_cache[split]
    for edge_type in [('node', 'to', 'node'), ('node', 'rev_to', 'node')]:
        attr = data[edge_type].edge_attr
        assert attr.shape[1] == expected_dim, (split, edge_type, attr.shape)
        assert torch.isfinite(attr).all(), (split, edge_type, 'non-finite feature')
    forward_attr = data['node', 'to', 'node'].edge_attr
    # History flags are columns 4+2 and 4+3, before the final port column.
    for column in [6, 7]:
        values = torch.unique(forward_attr[:, column])
        assert set(values.tolist()).issubset({0.0, 1.0}), (split, column, values)
    print(split, data, 'edge_dim=', forward_attr.shape[1])

audit_path = Path('/kaggle/working/H_history_leakage_audit.txt')
audit_path.write_text(
    'PASS: same-timestamp edges do not observe one another.\n'
    'PASS: history continuous features normalized with train statistics only.\n'
    f'PASS: expected edge dimension after Ports = {expected_dim}.\n',
    encoding='utf-8'
)
del dataset_cache
gc.collect()
print('Audit:', audit_path)


## 8. Huấn luyện seed 42

- Full mode dự kiến khoảng 2–3 giờ sau khi có cache; tiền xử lý lần đầu có thể cộng thêm 10–30 phút.
- Notebook ghi log ra file và in heartbeat mỗi phút.
- Không chạy seed 43–44 trước khi kiểm tra kết quả seed 42.


In [ ]:
import subprocess, time

LOG_PATH = Path(f'/kaggle/working/H_history_{RUN_MODE}.log')
cmd = [sys.executable, '-u', '-m', 'fraudGT.main',
       '--cfg', str(RUN_CFG), '--repeat', '1', '--gpu', str(GPU),
       'name_tag', TAG]
print('Command:', ' '.join(cmd))

started = time.time()
with LOG_PATH.open('w', encoding='utf-8') as stream:
    process = subprocess.Popen(cmd, cwd=repo, stdout=stream,
                               stderr=subprocess.STDOUT, text=True)
    while process.poll() is None:
        time.sleep(60)
        elapsed = (time.time() - started) / 60
        print(f'[heartbeat] {elapsed:.0f} min — H-FraudGT still running', flush=True)
        subprocess.run(['nvidia-smi', '--query-gpu=index,memory.used,utilization.gpu',
                        '--format=csv,noheader'], check=False)

if process.returncode != 0:
    tail = LOG_PATH.read_text(encoding='utf-8', errors='replace').splitlines()[-80:]
    print('\n'.join(tail))
    raise RuntimeError(f'H-FraudGT failed with exit code {process.returncode}')

print(f'Finished in {(time.time() - started) / 60:.1f} minutes')
print('Log:', LOG_PATH)
print('\n'.join(LOG_PATH.read_text(encoding='utf-8', errors='replace').splitlines()[-40:]))


## 9. Tổng hợp kết quả đúng protocol

Kết quả chính dùng threshold cố định 0.50. Kết quả chọn threshold bằng validation chỉ là phân tích bổ sung. Test không tham gia chọn epoch hoặc threshold.


In [ ]:
import pandas as pd

RUN_DIR = repo / 'results' / f'{RUN_CFG.stem}-{TAG}-gpu{GPU}'
summarizer = repo / 'scripts' / 'summarize_thresholds.py'
SUMMARY_FIXED = Path(f'/kaggle/working/summary_H_{RUN_MODE}_fixed_050.csv')
SUMMARY_SELECTED = Path(f'/kaggle/working/summary_H_{RUN_MODE}_val_selected.csv')

subprocess.run([sys.executable, str(summarizer), str(RUN_DIR),
                '--output', str(SUMMARY_FIXED), '--fixed-threshold', '0.50'], check=True)
subprocess.run([sys.executable, str(summarizer), str(RUN_DIR),
                '--output', str(SUMMARY_SELECTED)], check=True)

fixed = pd.read_csv(SUMMARY_FIXED)
fixed.insert(0, 'protocol', 'fixed_0.50_primary')
selected = pd.read_csv(SUMMARY_SELECTED)
selected.insert(0, 'protocol', 'validation_selected_secondary')
summary = pd.concat([fixed, selected], ignore_index=True)
SUMMARY_COMBINED = Path(f'/kaggle/working/summary_H_{RUN_MODE}.csv')
summary.to_csv(SUMMARY_COMBINED, index=False)
display(summary[['protocol', 'seed', 'best_epoch', 'threshold', 'val_f1',
                 'test_f1', 'test_precision', 'test_recall', 'test_auc']])
print('Run directory:', RUN_DIR)
print('Combined summary:', SUMMARY_COMBINED)


## 10. Đường học theo epoch


In [ ]:
import json
import matplotlib.pyplot as plt

def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines()
            if line.strip()]

train_rows = read_jsonl(RUN_DIR / str(SEED) / 'train' / 'stats.json')
val_rows = read_jsonl(RUN_DIR / str(SEED) / 'val' / 'stats.json')
test_rows = read_jsonl(RUN_DIR / str(SEED) / 'test' / 'stats.json')
epochs = [row['epoch'] for row in val_rows]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0, 0].plot([r['epoch'] for r in train_rows], [r['loss'] for r in train_rows], label='Train')
axes[0, 0].plot(epochs, [r['loss'] for r in val_rows], label='Validation')
axes[0, 0].plot(epochs, [r['loss'] for r in test_rows], label='Test')
axes[0, 0].set_title('Loss theo epoch')

axes[0, 1].plot(epochs, [r['f1_t50'] for r in val_rows], label='Validation F1')
axes[0, 1].plot(epochs, [r['f1_t50'] for r in test_rows], label='Test F1')
axes[0, 1].set_title('F1 tại threshold 0.50')

axes[1, 0].plot(epochs, [r['precision_t50'] for r in val_rows], label='Validation precision')
axes[1, 0].plot(epochs, [r['recall_t50'] for r in val_rows], label='Validation recall')
axes[1, 0].set_title('Precision và recall trên validation')

axes[1, 1].plot(epochs, [r['auc'] for r in val_rows], label='Validation ROC-AUC')
axes[1, 1].plot(epochs, [r['auc'] for r in test_rows], label='Test ROC-AUC')
axes[1, 1].set_title('ROC-AUC theo epoch')

for ax in axes.flat:
    ax.set_xlabel('Epoch'); ax.grid(alpha=0.25); ax.legend()
fig.tight_layout()
PLOT_PATH = Path(f'/kaggle/working/training_curves_H_{RUN_MODE}_seed42.png')
fig.savefig(PLOT_PATH, dpi=180, bbox_inches='tight')
plt.show()
print('Plot:', PLOT_PATH)


## 11. Tạo ghi chú và gói file gửi giảng viên

File ghi chú tự điền kết quả thực tế. Nếu `RUN_MODE=smoke`, ghi chú sẽ đánh dấu rõ đây chỉ là kiểm tra kỹ thuật, không phải kết quả khoa học.


In [ ]:
import shutil

primary = fixed.iloc[0]
status = ('Kết quả thử nghiệm đầy đủ một seed.' if RUN_MODE == 'full'
          else 'Smoke test kỹ thuật 2 epoch; chưa dùng để kết luận chất lượng mô hình.')
NOTE_PATH = Path(f'/kaggle/working/Ghi_chu_gui_thay_H_FraudGT_{RUN_MODE}.md')
NOTE_PATH.write_text(f'''# Tiến độ thử nghiệm H-FraudGT

## Mục tiêu

Kiểm tra riêng tác động của đặc trưng hành vi lịch sử trước khi thay đổi cơ chế lấy mẫu lân cận. FraudGT Encoder, RMP, Ports, Ego IDs, prediction head, loss và ngân sách huấn luyện được giữ nguyên so với baseline A.

## Thay đổi

Bổ sung 8 đặc trưng cạnh: khoảng thời gian từ lần gửi/nhận trước, cờ đã từng gửi/nhận, số giao dịch gửi/nhận trước, số giao dịch trước của cặp tài khoản và tỷ lệ số tiền hiện tại so với trung bình gửi trước đó. Mỗi đặc trưng chỉ dùng giao dịch có timestamp nhỏ hơn nghiêm ngặt; chuẩn hóa chỉ dùng thống kê tập train.

## Protocol

- Dataset: AML-Small-HI
- Seed: {SEED}
- Commit: `{commit}`
- Chọn epoch: validation F1
- Báo cáo chính: threshold cố định 0.50
- Trạng thái: {status}

## Kết quả chính

- Best validation epoch: {int(primary['best_epoch'])}
- Validation F1: {primary['val_f1']:.5f}
- Test F1: {primary['test_f1']:.5f}
- Test precision: {primary['test_precision']:.5f}
- Test recall: {primary['test_recall']:.5f}
- Test ROC-AUC: {primary['test_auc']:.5f}

Kết quả hiện tại chỉ gồm một seed, vì vậy chưa báo cáo mean ± standard deviation. Bước tiếp theo là so sánh cùng protocol với baseline A; chỉ chạy thêm seed 43–44 nếu H cho tín hiệu hợp lý trên validation.
''', encoding='utf-8')

BUNDLE = Path(f'/kaggle/working/H_FraudGT_{RUN_MODE}_artifacts')
BUNDLE.mkdir(exist_ok=True)
for path in [RUN_CFG, LOG_PATH, audit_path, SUMMARY_FIXED, SUMMARY_SELECTED,
             SUMMARY_COMBINED, PLOT_PATH, NOTE_PATH]:
    if path.exists():
        shutil.copy2(path, BUNDLE / path.name)
archive = shutil.make_archive(str(BUNDLE), 'zip', BUNDLE)
print(NOTE_PATH.read_text(encoding='utf-8'))
print('Tải file để gửi thầy:', archive)


## Sau khi chạy xong

1. Gửi file ZIP và ảnh biểu đồ, không chỉ gửi ảnh output notebook.
2. So sánh H với baseline A tại cùng threshold 0.50 và seed 42.
3. Nếu H không cải thiện validation F1/PR-AUC, chưa chạy thêm seed; kiểm tra đặc trưng và phân bố trước.
4. Nếu H có tín hiệu tốt, chạy seed 43–44 rồi mới triển khai temporal sampler T và mô hình TH.
